# E1 · W4 · Kaggle run 11 — tạo gói ảnh tập chuẩn CVAT (CPU)
**Mục đích:** chọn khoảng 200 frame ảnh khả kiến từ video train + val (không dùng test), tách thành gói ảnh cho người vẽ và khóa đáp án riêng (issue #24, ADR-0006).

**Trước khi chạy:**
1. Dataset `e1-w3-tools` → **New Version** thêm `w4_cvat_gold.py` (giữ `w3_export_yolo.py`, `video_split_v1.csv`, `annotation_audit.csv`, `review_assignment_w3.csv`).
2. *Add Input*: `e1-svanstrom-raw`, `e1-w3-tools` (bản mới). *Settings*: Accelerator **None**, Internet **On**. Điền `COMMIT`.

**Sau khi chạy:** tải `cvat_package.zip` (gửi An) và `gold_KEY.zip` (**Quang giữ, không gửi An, không đưa vào Git** cho tới khi An commit annotation), cùng `w4_cvat_gold/stage_status.json`, `w4_cvat_gold/gold_selection.json`.

In [ ]:
COMMIT = ""   # <-- SHA day du 40 ky tu

In [ ]:
import glob, hashlib, json, os, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu cua commit tren main chua cac script (GitHub Desktop -> History -> Copy SHA)"

In [ ]:
!pip -q install mcos-decoder
import mcos_decoder; print("mcos-decoder OK")

In [ ]:
# Tim thu muc du lieu tho; neu dataset tho la file .zip thi giai nen vao /kaggle/working/raw
if not glob.glob("/kaggle/input/**/Data/Video_V", recursive=True) and not glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True):
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        print("giai nen", z); zipfile.ZipFile(z).extractall("/kaggle/working/raw")
DATA_ROOT = "/kaggle/working/raw" if glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True) else "/kaggle/input"
assert glob.glob(f"{DATA_ROOT}/**/Data/Video_V", recursive=True), "Khong thay Data/Video_V — kiem tra input e1-svanstrom-raw"
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
TOOLS = {n: find(n) for n in ["w4_cvat_gold.py", "w3_export_yolo.py", "video_split_v1.csv", "annotation_audit.csv", "review_assignment_w3.csv"]}
assert os.path.dirname(TOOLS["w3_export_yolo.py"]) == os.path.dirname(TOOLS["w4_cvat_gold.py"])
for n, p in TOOLS.items():
    print(f"{n:28s} sha256={sha(p)[:16]}  {p}")

In [ ]:
!{sys.executable} {TOOLS["w4_cvat_gold.py"]} --data-root {DATA_ROOT} --split-csv {TOOLS["video_split_v1.csv"]} --audit {TOOLS["annotation_audit.csv"]} --review {TOOLS["review_assignment_w3.csv"]} --out /kaggle/working --commit {COMMIT}

## Tự kiểm tra (phải in `PASS`)

In [ ]:
st = json.load(open("/kaggle/working/w4_cvat_gold/stage_status.json")); print({k: v["status"] for k, v in st.items()})
print(st.get("select")); print(st.get("package"))
ok = len(st) == 4 and all(v["status"] == "OK" for v in st.values()) and st["select"]["frames"] >= 150
print("PASS" if ok else "FAIL - gui stage_status.json cho Claude")